In [48]:
"""
Interactive Structural Validation Suite: AlphaFold vs. Experimental PDB
Author: Miri Krupkin, PhD
Description: End-to-end publication-grade bioinformatics pipeline featuring
one-click HIV-1 RTase demo mode, automated construct trimming, True Sequence
Alignment mapping (Needleman-Wunsch) to bypass crystal gaps, sequence identity
validation, and synchronized 3D visualization.
"""

# Auto-install missing dependencies if running in a clean environment
try:
    import Bio
    import pandas
    import py3Dmol
except ImportError:
    print("[INFO] Required packages not found. Installing dependencies...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "biopython", "pandas", "py3Dmol"])
    print("[INFO] Dependencies successfully installed!")

import os
import json
import pandas as pd
from Bio import Align
from Bio.PDB import PDBParser, Superimposer, ShrakeRupley, PDBIO, PDBList
from Bio.PDB.MMCIFParser import MMCIFParser
from urllib.request import urlopen
from IPython.display import display, HTML

# Detect if running in Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Amino acid 3-to-1 letter mapping for robust sequence alignment
D3TO1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
         'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
         'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
         'ALA': 'A', 'VAL':'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

def fetch_alphafold_prediction(uniprot_id):
    url = f"https://alphafold.com/api/prediction/{uniprot_id}"
    try:
        with urlopen(url) as response:
            data = json.loads(response.read().decode())
            if isinstance(data, list) and len(data) > 0:
                entry = data[0]
                return entry.get('pdbUrl'), entry.get('uniprotDescription', f'AlphaFold Model ({uniprot_id})')
    except Exception as e:
        print(f"Error fetching AlphaFold data for UniProt {uniprot_id}: {e}")
    return None, f"UniProt Model ({uniprot_id})"

def download_file(url, filename):
    with urlopen(url) as response, open(filename, 'wb') as f:
        f.write(response.read())

def download_experimental_pdb(pdb_code, output_filename):
    try:
        pdbl = PDBList()
        downloaded_path = pdbl.retrieve_pdb_file(pdb_code.lower(), pdir='.', file_format='pdb')
        if os.path.exists(downloaded_path):
            if os.path.exists(output_filename):
                os.remove(output_filename)
            os.rename(downloaded_path, output_filename)
            return True
    except Exception as e:
        print(f"[WARNING] Biopython PDB download error: {e}")
    return False

def parse_structure_file(filepath, structure_id="model"):
    ext = os.path.splitext(filepath)[1].lower()
    parser = MMCIFParser(QUIET=True) if ext == ".cif" else PDBParser(QUIET=True)
    return parser.get_structure(structure_id, filepath)

def analyze_alphafold_model(pdb_path):
    structure = parse_structure_file(pdb_path, "af_model")
    sr = ShrakeRupley()
    sr.compute(structure, level="R")

    records = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if "CA" in residue:
                    records.append({
                        "Residue_ID": residue.get_id()[1],
                        "Residue": residue.get_resname(),
                        "pLDDT": residue["CA"].get_bfactor(),
                        "SASA": residue.sasa
                    })
    return pd.DataFrame(records)

def calculate_structural_rmsd_and_deviation(ref_pdb_path, target_pdb_path, trim_excess=False):
    """Aligns structures using True Sequence Alignment (bypassing internal gaps/numbering offsets)."""
    ref_structure = parse_structure_file(ref_pdb_path, "experimental")
    target_structure = parse_structure_file(target_pdb_path, "predicted")

    ref_chains = sorted([chain for model in ref_structure for chain in model], key=lambda c: len(list(c.get_atoms())), reverse=True)
    target_chains = sorted([chain for model in target_structure for chain in model], key=lambda c: len(list(c.get_atoms())), reverse=True)

    ref_atoms_matched = []
    target_atoms_matched = []
    trimmed_info = "No trimming applied (Full-length alignment)."

    aligner = Align.PairwiseAligner()
    aligner.mode = 'global'
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5

    target_matched_ids = set()

    for r_chain, t_chain in zip(ref_chains, target_chains):
        r_list = [res['CA'] for res in r_chain if 'CA' in res]
        t_list = [res['CA'] for res in t_chain if 'CA' in res]

        # Extract 1-letter sequences
        seq_r = "".join([D3TO1.get(res.get_parent().get_resname(), 'X') for res in r_list])
        seq_t = "".join([D3TO1.get(res.get_parent().get_resname(), 'X') for res in t_list])

        # Perform dynamic sequence alignment to perfectly map atoms across gaps
        alignments = aligner.align(seq_r, seq_t)
        best_alignment = alignments[0]

        for (r_start, r_end), (t_start, t_end) in zip(best_alignment.aligned[0], best_alignment.aligned[1]):
            for i in range(r_end - r_start):
                ref_atoms_matched.append(r_list[r_start + i])
                matched_t_atom = t_list[t_start + i]
                target_atoms_matched.append(matched_t_atom)
                target_matched_ids.add(matched_t_atom.get_parent().get_id())

        # Handle structural detaching of unaligned residues (Construct Trimming)
        if trim_excess and len(t_list) > len(r_list):
            trimmed_info = "Construct Trimming: Unaligned terminal tags and loops dynamically pruned to match crystal construct."
            for res in list(t_chain):
                if res.has_id('CA') and res.get_id() not in target_matched_ids:
                    res.get_parent().detach_child(res.get_id())

    min_len = len(ref_atoms_matched)

    # 1. Apply Superimposer rotation to target structure FIRST
    superimposer = Superimposer()
    superimposer.set_atoms(ref_atoms_matched, target_atoms_matched)
    superimposer.apply(target_structure.get_atoms())

    # 2. Calculate true sequence identity and deviations using exactly paired atoms
    identical_count = 0
    deviation_records = []
    for r_atom, t_atom in zip(ref_atoms_matched, target_atoms_matched):
        if r_atom.get_parent().get_resname() == t_atom.get_parent().get_resname():
            identical_count += 1

        diff_vector = r_atom.get_coord() - t_atom.get_coord()
        distance = float((diff_vector ** 2).sum() ** 0.5)

        res = t_atom.get_parent()
        deviation_records.append({
            "Residue_ID": res.get_id()[1],
            "Residue": res.get_resname(),
            "Structural_Deviation_Angstroms": round(distance, 3)
        })

    seq_identity = (identical_count / min_len) * 100 if min_len > 0 else 0

    aligned_filename = "aligned_prediction.pdb"
    io = PDBIO()
    io.set_structure(target_structure)
    io.save(aligned_filename)

    return superimposer.rms, min_len, seq_identity, aligned_filename, pd.DataFrame(deviation_records), trimmed_info

if __name__ == "__main__":
    print("==================================================")
    print("   STRUCTURAL INTEGRITY & VALIDATION PIPELINE     ")
    print("==================================================")

    demo_choice = input("Run Quick Demo (HIV-1 RTase Heterodimer with Auto-Trimming)? [Y/n]: ").strip().lower()

    target_filename = None
    target_name = "HIV-1 RT Heterodimer"
    identifier_desc = "Default GitHub Custom CIF Model"
    exp_code = "1REV"
    should_trim = True

    DEFAULT_PRED_URL = "https://raw.githubusercontent.com/mirikrupkin/structural-ai-validation-suite/refs/heads/main/1rev_prediction.cif"

    if demo_choice in ['', 'y', 'yes']:
        print("\n[DEMO MODE] Initializing automated validation on HIV-1 RTase...")
        target_filename = "1rev_prediction.cif"
        try:
            download_file(DEFAULT_PRED_URL, target_filename)
        except Exception as e:
            print(f"[ERROR] Could not download default prediction: {e}")

        exp_filename = f"EXP_{exp_code}.pdb"
        print(f"[DEMO MODE] Downloading experimental reference PDB {exp_code}...")
        download_experimental_pdb(exp_code, exp_filename)

        print(f"[DEMO MODE] Running True Sequence-Aligned spatial mapping with construct trimming against {exp_code}...")
        rmsd_val, matched_res, seq_id, aligned_pred_file, df_dev, trim_report = calculate_structural_rmsd_and_deviation(exp_filename, target_filename, trim_excess=should_trim)

    else:
        print("\nSelect Prediction Model Source:")
        print("  [1] Default HIV-1 RT Heterodimer (Demo)")
        print("  [2] AlphaFold API by UniProt ID (e.g., P00698)")
        print("  [3] Upload your own structure file (.pdb / .cif) or enter URL")
        selection = input("Enter choice [1, 2, or 3]: ").strip()

        if selection == '1':
            target_filename = "1rev_prediction.cif"
            target_name = "HIV-1 RT Heterodimer"
            identifier_desc = "Default GitHub Custom CIF Model"
            exp_code = "1REV"
            print(f"\n[INFO] Downloading default custom prediction from GitHub...")
            download_file(DEFAULT_PRED_URL, target_filename)

        elif selection == '2':
            uniprot_id = input("Enter AlphaFold UniProt ID (e.g., P00698): ").strip() or "P00698"
            print(f"\n1. Fetching AlphaFold Computed Structure Model for UniProt ID {uniprot_id}...")
            af_url, protein_title = fetch_alphafold_prediction(uniprot_id)
            target_name = protein_title
            exp_code = "1IEE" if uniprot_id.upper() == "P00698" else "1UBQ"
            if af_url:
                target_filename = f"AF_{uniprot_id}.pdb"
                download_file(af_url, target_filename)
                identifier_desc = f"UniProt API: {uniprot_id} | {protein_title}"
        else:
            print("\n[INFO] Drag and drop your prediction file (.pdb or .cif) into the left-hand Colab file panel (folder icon).")
            target_filename = input("Enter the exact filename of your uploaded file (e.g., model.pdb): ").strip()
            target_name = input("Enter descriptive protein name for your model: ").strip()
            identifier_desc = f"User Uploaded File: {target_filename}"
            exp_code = "1UBQ"

        if target_filename and os.path.exists(target_filename):
            print(f"--> Successfully loaded prediction model: {target_name} ({target_filename})")
            has_exp = input("\nUse validation workflow with experimental reference? [Y/n]: ").strip().lower()
            if has_exp in ['', 'y', 'yes']:
                user_input_pdb = input(f"Enter 4-letter PDB code (suggested: {exp_code}): ").strip() or exp_code
                exp_filename = f"EXP_{user_input_pdb.upper()}.pdb"
                print(f"--> Downloading PDB {user_input_pdb.upper()} using Biopython PDBList...")
                success = download_experimental_pdb(user_input_pdb, exp_filename)
                if success and os.path.exists(exp_filename):
                    print(f"--> Successfully downloaded and saved {exp_filename}")
                    trim_choice = input("\n[Construct Design Feature] Would you like to automatically trim unaligned regions/tags? [y/N]: ").strip().lower()
                    should_trim = trim_choice in ['y', 'yes']
                    print(f"--> Running True Sequence-Aligned spatial mapping against: {exp_filename}")
                    rmsd_val, matched_res, seq_id, aligned_pred_file, df_dev, trim_report = calculate_structural_rmsd_and_deviation(exp_filename, target_filename, trim_excess=should_trim)
                else:
                    exp_filename = None
            else:
                exp_filename = None

    if target_filename and os.path.exists(target_filename):
        df_metrics = analyze_alphafold_model(target_filename)
        mean_plddt = df_metrics["pLDDT"].mean()
        pct_very_high = (df_metrics["pLDDT"] > 90).mean() * 100
        pct_confident = ((df_metrics["pLDDT"] <= 90) & (df_metrics["pLDDT"] > 70)).mean() * 100

        if 'exp_filename' in locals() and exp_filename and os.path.exists(exp_filename) and 'rmsd_val' in locals():
            print(f"\n--- Benchmark Results ---")
            print(f"Experimental Reference PDB: {exp_filename}")
            print(f"Matched Alpha-Carbon Residues: {matched_res}")
            print(f"Sequence Identity: {seq_id:.1f}%")
            if seq_id > 80: print(f"--> Prediction and compared PDB have >80% identity. This is a good PDB for structure validation.")

            print(f"Global Coordinate RMSD: {rmsd_val:.3f} Å")
            print(f"Construct Status: {trim_report}")
            print(f"\n--- Top 5 Most Structurally Divergent Residues ---")
            print(df_dev.sort_values(by="Structural_Deviation_Angstroms", ascending=False).head(5).to_string(index=False))
        else:
            aligned_pred_file, df_dev, exp_filename = None, None, None

        df_metrics["Target_Name"] = target_name
        if df_dev is not None: df_metrics = pd.merge(df_metrics, df_dev, on=["Residue_ID", "Residue"], how="left")
        df_metrics.to_csv("validation_metrics_output.csv", index=False)

        print(f"\n--- AlphaFold Structural Confidence Summary ---")
        print(f"Mean Complex pLDDT Score: {mean_plddt:.2f} / 100")
        print(f"  • Very High Confidence (>90): {pct_very_high:.1f}%")

        display(HTML(f"""
        <div style="font-family: sans-serif; margin-bottom: 8px; border-bottom: 2px solid #3498db; padding-bottom: 5px;">
            <div style="display: flex; justify-content: space-around;">
                <div style="text-align: center; width: 48%;">
                    <h3 style="margin: 0; color: #2c3e50;">Aligned Prediction Model</h3>
                    <p style="margin: 3px 0; font-size: 13px; color: #2980b9;"><b>Protein:</b> {target_name}</p>
                    <p style="margin: 0; font-size: 11px; color: #7f8c8d;">{trim_report if 'trim_report' in locals() else 'Standard View'}</p>
                </div>
                <div style="text-align: center; width: 48%;">
                    <h3 style="margin: 0; color: #2c3e50;">Experimental Reference PDB</h3>
                    <p style="margin: 3px 0; font-size: 13px; color: #27ae60;"><b>PDB Source:</b> {exp_filename if exp_filename else 'None'}</p>
                    <p style="margin: 0; font-size: 11px; color: #7f8c8d;">Synchronized Orientation View (Cyan)</p>
                </div>
            </div>
        </div>
        """))

        left_file = aligned_pred_file if (aligned_pred_file and os.path.exists(aligned_pred_file)) else target_filename
        with open(left_file, 'r') as f: left_data = f.read()
        exp_data = open(exp_filename, 'r').read() if exp_filename else left_data

        try:
            viewer = py3Dmol.view(viewergrid=(1, 2), width=920, height=450)
            viewer.addModel(left_data, "pdb" if left_file.endswith(".pdb") else "cif", viewer=(0, 0))
            viewer.setStyle({'viewer': (0, 0)}, {'cartoon': {'color': 'lightblue'}})
            viewer.addModel(exp_data, 'pdb', viewer=(0, 1))
            viewer.setStyle({'viewer': (0, 1)}, {'cartoon': {'color': 'cyan'}})
            viewer.zoomTo()
            viewer.show()
        except Exception: pass

   STRUCTURAL INTEGRITY & VALIDATION PIPELINE     
Run Quick Demo (HIV-1 RTase Heterodimer with Auto-Trimming)? [Y/n]: n

Select Prediction Model Source:
  [1] Default HIV-1 RT Heterodimer (Demo)
  [2] AlphaFold API by UniProt ID (e.g., P00698)
  [3] Upload your own structure file (.pdb / .cif) or enter URL
Enter choice [1, 2, or 3]: 1

[INFO] Downloading default custom prediction from GitHub...
--> Successfully loaded prediction model: HIV-1 RT Heterodimer (1rev_prediction.cif)

Use validation workflow with experimental reference? [Y/n]: y
Enter 4-letter PDB code (suggested: 1REV): 1REV
--> Downloading PDB 1REV using Biopython PDBList...
--> Successfully downloaded and saved EXP_1REV.pdb

[Construct Design Feature] Would you like to automatically trim unaligned regions/tags? [y/N]: y
--> Running True Sequence-Aligned spatial mapping against: EXP_1REV.pdb

--- Benchmark Results ---
Experimental Reference PDB: EXP_1REV.pdb
Matched Alpha-Carbon Residues: 936
Sequence Identity: 100.0%
-->

3Dmol.js failed to load for some reason. Please check your browser console for error messages.